# CropHist Iowa Sample

This notebook uses the published Iowa sample directly from CropHist public URLs.

- no API key required
- 100,000 Iowa fields in the published sample
- 2008-2025 crop history in the current export

To keep the notebook readable, the walkthrough below uses a small set of sample points and shows the full 18-year history for each one.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

SAMPLE_BASE = 'https://crophist.com/pages/iowa-sample/downloads'
CROP_COLORS = {
    'Corn': '#f4c430',
    'Soybeans': '#43a047',
    'Grassland/Pasture': '#7cb342',
    'Developed/Open_Space': '#8d6e63',
    'No data': '#94a3b8',
    'Unknown': '#94a3b8',
}


def point_label(lon, lat):
    return f'{lon:.4f}, {lat:.4f}'


def crop_color(crop_name):
    return CROP_COLORS.get(crop_name, '#94a3b8')


def crop_chip(crop_name):
    label = crop_name if pd.notna(crop_name) and str(crop_name).strip() else 'No data'
    return (
        '<span style="white-space:nowrap;">'
        f'<span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{crop_color(label)};margin-right:6px;vertical-align:middle;"></span>'
        f'{label}'
        '</span>'
    )


def render_crop_legend(crops):
    items = ''.join(
        f'<span style="display:inline-block;margin:0 14px 10px 0;">{crop_chip(crop)}</span>'
        for crop in crops
    )
    display(HTML(f'<div style="margin:8px 0 14px;">{items}</div>'))


def render_crop_matrix(matrix):
    html_df = matrix.fillna('No data').map(crop_chip)
    html = html_df.to_html(escape=False)
    html = html.replace('<table border="1" class="dataframe">', '<table class="crop-matrix">')
    style = '''
    <style>
      table.crop-matrix {
        border-collapse: collapse;
        font-family: Arial, sans-serif;
        font-size: 14px;
      }
      table.crop-matrix th,
      table.crop-matrix td {
        border-bottom: 1px solid #e5e7eb;
        padding: 8px 10px;
        text-align: left;
      }
      table.crop-matrix thead th {
        background: #f8fafc;
      }
      table.crop-matrix tbody tr:nth-child(even) {
        background: #f8fafc;
      }
    </style>
    '''
    display(HTML(style + html))


In [ ]:
all_history = pd.read_parquet(f'{SAMPLE_BASE}/iowa-fields-sample-long.parquet')
sample_points = (
    all_history[['field_id', 'lon', 'lat']]
    .drop_duplicates()
    .sort_values(['lon', 'lat'])
    .head(4)
    .reset_index(drop=True)
)
sample_points['point'] = sample_points.apply(lambda row: point_label(row['lon'], row['lat']), axis=1)

history_df = all_history[all_history['field_id'].isin(sample_points['field_id'])].copy()
history_df['point'] = history_df.apply(lambda row: point_label(row['lon'], row['lat']), axis=1)
history_df = history_df.sort_values(['point', 'year']).reset_index(drop=True)

year_min = int(history_df['year'].min())
year_max = int(history_df['year'].max())
year_count = history_df['year'].nunique()

print(
    f'Loaded {len(history_df):,} rows across {sample_points.shape[0]} Iowa sample points '
    f'and {year_count} years ({year_min}-{year_max}).'
)
display(sample_points[['lon', 'lat']])


In [ ]:
crop_matrix = history_df.pivot(index='year', columns='point', values='crop_name').sort_index()
present_crops = sorted(history_df['crop_name'].dropna().unique())

render_crop_legend(present_crops)
render_crop_matrix(crop_matrix)


In [ ]:
example_point = sample_points['point'].iloc[0]
example = history_df[history_df['point'] == example_point].sort_values('year').copy()

fig, ax = plt.subplots(figsize=(12, 2.6))
ax.scatter(
    example['year'],
    [1] * len(example),
    s=320,
    c=example['crop_name'].map(crop_color),
    edgecolors='#111827',
    linewidths=1.2,
)
ax.set_title(f'Rotation timeline for {example_point}')
ax.set_xlabel('Year')
ax.set_xticks(example['year'])
ax.set_xlim(example['year'].min() - 0.6, example['year'].max() + 0.6)
ax.set_yticks([])
ax.grid(axis='x', alpha=0.25)
for spine in ['left', 'right', 'top']:
    ax.spines[spine].set_visible(False)
plt.show()


In [ ]:
crop_frequency = (
    history_df.groupby(['point', 'crop_name'])
    .size()
    .reset_index(name='years_as_top_crop')
    .sort_values(['point', 'years_as_top_crop', 'crop_name'], ascending=[True, False, True])
)
crop_frequency_text = (
    crop_frequency.groupby('point')
    .apply(
        lambda group: ', '.join(
            f"{row.crop_name} ({row.years_as_top_crop})" for row in group.itertuples()
        )
    )
    .rename('crops_by_frequency')
    .reset_index()
)

rotation_summary = (
    history_df.sort_values('year')
    .assign(is_corn_soy=lambda df: df['crop_name'].isin(['Corn', 'Soybeans']))
    .groupby(['point', 'lon', 'lat'], as_index=False)
    .agg(
        years=('year', 'nunique'),
        last_crop=('crop_name', 'last'),
        corn_soy_share=('is_corn_soy', 'mean'),
    )
    .merge(crop_frequency_text, on='point', how='left')
    .sort_values(['lon', 'lat'])
)
rotation_summary['corn_soy_share_pct'] = (rotation_summary['corn_soy_share'] * 100).round(1)

display(
    rotation_summary[
        ['lon', 'lat', 'years', 'last_crop', 'crops_by_frequency', 'corn_soy_share_pct']
    ]
)


In [ ]:
def baseline_forecast(last_crop):
    if last_crop == 'Corn':
        return 'Soybeans'
    if last_crop == 'Soybeans':
        return 'Corn'
    return last_crop


forecast_table = rotation_summary.copy()
forecast_table['forecast_2026'] = forecast_table['last_crop'].map(baseline_forecast)

display(
    forecast_table[
        ['lon', 'lat', 'last_crop', 'forecast_2026', 'crops_by_frequency']
    ]
)

forecast_mix = (
    forecast_table['forecast_2026']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename('share_pct')
    .reset_index()
    .rename(columns={'index': 'forecast_2026'})
)
display(forecast_mix)
